In [1]:
import torch

if torch.cuda.is_available():
    x = torch.randn(3, 3, device='cuda')
    y = torch.matmul(x, x.T)
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('Tensor device:', x.device)
    print('Matrix multiply result shape:', y.shape)
else:
    print('CUDA is not available in this notebook environment.')


CUDA device: NVIDIA GeForce MX350
Tensor device: cuda:0
Matrix multiply result shape: torch.Size([3, 3])


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce MX350
CUDA version: 12.1


### Load PDFs

In [3]:
from pathlib import Path
from pypdf import PdfReader

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "raw"

pdf_files = list(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")

for pdf in pdf_files:
    print(f"- {pdf.name}")

Found 3 PDF files:
- Session 3 intro to ML.pdf
- Session 6 - Linear Regresssion.pdf
- Session 7 Ml -logistic regression.pdf


### Load & Inspect

In [4]:
from pypdf import PdfReader

documents = []

for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    documents.append({
        "source": pdf_path.name,
        "pages": len(reader.pages),
        "text": text
    })

for doc in documents:
    print("=" * 80)
    print(f"Source: {doc['source']}")
    print(f"Pages: {doc['pages']}")
    print(f"Characters: {len(doc['text'])}")
    print("\nFirst 500 characters:")
    print(doc["text"][:500])

Source: Session 3 intro to ML.pdf
Pages: 40
Characters: 16413

First 500 characters:
Introduction to 
Machine 
Learning (ML)
Learning Objectives:
By the end of this session, you should: 
• Understand the Core Fields of AI – Learn about Machine Learning 
(ML), Deep Learning (DL), NLP, Computer Vision, Robotics, and 
Generative Models.
• Differentiate Between AI Subfields – Recognize how ML, DL, NLP, 
and Computer Vision contribute to AI advancements.
• Explore Real-World Applications of AI – Discover how AI is used in 
self-driving cars, chatbots, precision agriculture, personal 
Source: Session 6 - Linear Regresssion.pdf
Pages: 23
Characters: 2572

First 500 characters:
Machine 
Learning 
Session 6
Linear Regression
Presented By Eng : Mohamed Khaled 
WHAT IS LINEAR REGRESSION? 

WHAT IS LINEAR REGRESSION? 

Our hypothesis function has the general form
• like the equation of a straight line 
• create a function called 𝒉𝜽 that is trying to map 
our input data (the 𝑥′𝑠) to our output data

### Cleaning + Chunking ✂️

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["text"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "chunk_id": i
        })

print(f"Total chunks: {len(chunks)}")

Total chunks: 45


In [6]:
for chunk in chunks[:5]:
    print("=" * 80)
    print(f"Source: {chunk['source']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(chunk["text"][:800])

Source: Session 3 intro to ML.pdf
Chunk ID: 0
Introduction to 
Machine 
Learning (ML)
Learning Objectives:
By the end of this session, you should: 
• Understand the Core Fields of AI – Learn about Machine Learning 
(ML), Deep Learning (DL), NLP, Computer Vision, Robotics, and 
Generative Models.
• Differentiate Between AI Subfields – Recognize how ML, DL, NLP, 
and Computer Vision contribute to AI advancements.
• Explore Real-World Applications of AI – Discover how AI is used in 
self-driving cars, chatbots, precision agriculture, personal 
assistants, and more.
• See the Connection Between AI Technologies – Understand how AI, 
ML, DL, and Generative Models are interrelated and drive 
innovation.
What is Artificial Intelligence (AI)?
Definition
Artificial Intelligence (AI) is the science of making machines
Source: Session 3 intro to ML.pdf
Chunk ID: 1
innovation.
What is Artificial Intelligence (AI)?
Definition
Artificial Intelligence (AI) is the science of making machines 
think, lear

### Metadata + Embeddings

In [7]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded!")

Device: cuda
Embedding model loaded!


### Generate Embeddings

In [8]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (45, 384)


### Create ChromaDB

In [9]:
import chromadb

VECTOR_DB_DIR = PROJECT_DIR / "backend" / "data" / "vector_store"

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR)
)

collection = client.get_or_create_collection(
    name="studymate_documents",
    metadata={"hnsw:space": "cosine"}
)

print("Chroma collection ready!")

Chroma collection ready!


### Store Chunks + Embeddings

In [10]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"]
        }
        for chunk in chunks
    ]
)

print("Stored chunks:", collection.count())

Stored chunks: 45


### Test Retrieval

In [11]:
query = "What is linear regression?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

for i in range(3):
    print("=" * 80)
    print("Source:", results["metadatas"][0][i]["source"])
    print("Chunk ID:", results["metadatas"][0][i]["chunk_id"])
    print("Distance:", results["distances"][0][i])
    print(results["documents"][0][i][:800])

Source: Session 6 - Linear Regresssion.pdf
Chunk ID: 0
Distance: 0.18247240781784058
Machine 
Learning 
Session 6
Linear Regression
Presented By Eng : Mohamed Khaled 
WHAT IS LINEAR REGRESSION? 

WHAT IS LINEAR REGRESSION?
Source: Session 6 - Linear Regresssion.pdf
Chunk ID: 1
Distance: 0.4174213409423828
Our hypothesis function has the general form
• like the equation of a straight line 
• create a function called 𝒉𝜽 that is trying to map 
our input data (the 𝑥′𝑠) to our output data (the 𝑦′𝑠).
Linear Regression The Hypothesis Function
intercept and slope
Example
Interactive Visualization of Linear Regression (Demo )
Multiple variables (Features) 
• Linear regression with multiple variables is also known as "multivariate linear regression". 
Multiple variables (Features) 
Interactive Visualization of Linear Regression (Demo )
Cost Function
• We can measure the accuracy of our hypothesis function by using a cost 
function. 
• This takes an average of all the results of the hypothesis wi

In [12]:
def retrieve_documents(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )

    retrieved = []

    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "chunk_id": results["metadatas"][0][i]["chunk_id"],
            "distance": results["distances"][0][i]
        })

    return retrieved

In [13]:
retrieved = retrieve_documents(
    "What is the hypothesis function in linear regression?",
    k=3
)

for doc in retrieved:
    print("=" * 80)
    print("Source:", doc["source"])
    print("Distance:", doc["distance"])
    print(doc["text"][:500])

Source: Session 6 - Linear Regresssion.pdf
Distance: 0.26666587591171265
Our hypothesis function has the general form
• like the equation of a straight line 
• create a function called 𝒉𝜽 that is trying to map 
our input data (the 𝑥′𝑠) to our output data (the 𝑦′𝑠).
Linear Regression The Hypothesis Function
intercept and slope
Example
Interactive Visualization of Linear Regression (Demo )
Multiple variables (Features) 
• Linear regression with multiple variables is also known as "multivariate linear regression". 
Multiple variables (Features) 
Interactive Visualizatio
Source: Session 6 - Linear Regresssion.pdf
Distance: 0.47825127840042114
Machine 
Learning 
Session 6
Linear Regression
Presented By Eng : Mohamed Khaled 
WHAT IS LINEAR REGRESSION? 

WHAT IS LINEAR REGRESSION?
Source: Session 7 Ml -logistic regression.pdf
Distance: 0.5863596200942993
Machine 
Learning 
Session
Logistic regression 
Presented by Eng : Mohamed Khaled 
Classification 
Email: Spam / Non Spam
Online transaction

### : Grounded Generation

In [16]:
from ollama import chat

def build_context(retrieved_docs):
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        context_parts.append(
            f"[Source {i}: {doc['source']} | Chunk {doc['chunk_id']}]\n"
            f"{doc['text']}"
        )

    return "\n\n".join(context_parts)

In [17]:
def generate_answer(question, k=3):
    # Retrieve relevant documents
    retrieved_docs = retrieve_documents(question, k=k)

    # Build context from retrieved chunks
    context = build_context(retrieved_docs)

    prompt = f"""
You are StudyMate, a university study assistant.

Answer the student's question using ONLY the provided course material.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not available in the provided context, say:
"I couldn't find this information in the provided course materials."
4. Keep the answer clear and educational.
5. At the end, list the sources used.

COURSE MATERIAL:
{context}

STUDENT QUESTION:
{question}
"""

    response = chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"], retrieved_docs

In [18]:
answer, sources = generate_answer(
    "What is the hypothesis function in linear regression?"
)

print(answer)

 The hypothesis function in linear regression is a function that maps the input data (x's) to the output data (y's), taking the form of a straight line. It is denoted by 𝒉𝜽 and tries to replicate the relationship between the input variables and the output variable. In other words, it creates a model that best fits the data points in a linear way.

In the provided course material, it is mentioned that linear regression can handle multiple variables (features) as well.

[Source 1: Session 6 - Linear Regresssion.pdf | Chunk 1, Chunk 0]
[Source 2: Session 6 - Linear Regresssion.pdf | Chunk 0]


In [19]:
answer, sources = generate_answer(
    "What is logistic regression used for?"
)

print(answer)

 Logistic regression is used for binary classification problems, where the goal is to predict the probability of an event occurring or not, based on a set of input features. In the context provided, logistic regression is used to predict whether a student passes or fails an exam based on their study hours.

[Sources: Session 7 Ml -logistic regression.pdf | Chunk 6, Chunk 7]


In [20]:
answer, sources = generate_answer(
    "What is reinforcement learning?"
)


print(answer)

 Reinforcement Learning is a type of Machine Learning where the model learns by interacting with an environment and receiving rewards or penalties. This learning method allows the model to make decisions to maximize the total reward over time. Examples of applications include teaching robots to walk, AI playing games like Chess and Go, and training agents for navigation tasks. Algorithms for Reinforcement Learning include Q-Learning, Deep Q Networks (DQN), and Proximal Policy Optimization (PPO). [Source 1: Session 3 intro to ML.pdf | Chunk 11]


In [21]:
test_questions = [
    "What is linear regression?",
    "What is the hypothesis function in linear regression?",
    "What is the cost function?",
    "What is multiple linear regression?",
    "What is logistic regression?",
    "What is the sigmoid function?",
    "What is classification?",
    "What is the difference between linear regression and logistic regression?",
    "What is overfitting?",
    "What is reinforcement learning?"
]

for i, question in enumerate(test_questions, start=1):
    print("=" * 100)
    print(f"Question {i}: {question}")
    
    answer, sources = generate_answer(question)
    
    print("\nAnswer:")
    print(answer)

Question 1: What is linear regression?

Answer:
 Linear Regression is a statistical method used to establish a relationship between two variables. The hypothesis function in linear regression is designed to map input data (x's) to output data (y's), and it has the form of an equation for a straight line. This equation includes an intercept (also known as the bias) and a slope (also known as the weight). Linear regression can also handle multiple variables (features). To measure the accuracy of the hypothesis function, a cost function is used, which takes an average of all the results of the hypothesis with inputs from x's compared to the actual output y's.

Sources:
1. Session 6 - Linear Regression.pdf | Chunk 0, 1, 5
Question 2: What is the hypothesis function in linear regression?

Answer:
 The hypothesis function in linear regression is a function that attempts to map our input data (the x's) to our output data (the y's). It has the general form of a straight line equation and is re

In [22]:
answer, sources = generate_answer(
    "Who won the FIFA World Cup in 2022?"
)

print(answer)

 I couldn't find this information in the provided course materials as they focus on Artificial Intelligence and Machine Learning, not sports or world events.


### Evaluation

In [23]:
evaluation_questions = [
    "What is machine learning?",
    "What are the main types of machine learning?",
    "What is supervised learning?",
    "What is linear regression?",
    "What is the hypothesis function in linear regression?",
    "What is the cost function?",
    "What is multiple linear regression?",
    "What is logistic regression?",
    "What is the sigmoid function?",
    "What is classification?"
]

evaluation_results = []

for i, question in enumerate(evaluation_questions, start=1):

    answer, sources = generate_answer(question)

    evaluation_results.append({
        "question_id": i,
        "question": question,
        "answer": answer,
        "sources": ", ".join(
            sorted(set(source["source"] for source in sources))
        )
    })

print(f"Evaluated {len(evaluation_results)} questions.")

Evaluated 10 questions.


In [24]:
import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df[
    ["question_id", "question", "sources"]
]

,question_id,question,sources
0,1,What is machine learning?,Session 3 intro to ML.pdf
1,2,What are the main types of machine learning?,Session 3 intro to ML.pdf
2,3,What is supervised learning?,Session 3 intro to ML.pdf
3,4,What is linear regression?,Session 6 - Linear Regresssion.pdf
4,5,What is the hypothesis function in linear regr...,"Session 6 - Linear Regresssion.pdf, Session 7 ..."
5,6,What is the cost function?,"Session 6 - Linear Regresssion.pdf, Session 7 ..."
6,7,What is multiple linear regression?,Session 6 - Linear Regresssion.pdf
7,8,What is logistic regression?,Session 7 Ml -logistic regression.pdf
8,9,What is the sigmoid function?,Session 7 Ml -logistic regression.pdf
9,10,What is classification?,Session 3 intro to ML.pdf


In [25]:
for row in evaluation_results:
    print("=" * 100)
    print(f"Q{row['question_id']}: {row['question']}")
    print("\nAnswer:")
    print(row["answer"])
    print("\nSources:")
    print(row["sources"])

Q1: What is machine learning?

Answer:
 Machine Learning (ML) is a branch of Artificial Intelligence (AI) that focuses on developing systems capable of learning from data without being explicitly programmed. Instead of writing detailed instructions for every possible scenario, ML models analyze patterns in data to make intelligent decisions.

[Source 1: Session 3 intro to ML.pdf | Chunk 10]
[Source 2: Session 3 intro to ML.pdf | Chunk 9]
[Source 3: Session 3 intro to ML.pdf | Chunk 0]

Sources:
Session 3 intro to ML.pdf
Q2: What are the main types of machine learning?

Answer:
 The main types of Machine Learning (ML) are Supervised Learning and Unsupervised Learning, as mentioned in the provided course materials.

Supervised Learning involves training a model using labeled data (input-output pairs) such as predicting house prices based on features like size and location. Algorithms for Supervised Learning include Linear Regression, Decision Trees, Random Forest, Support Vector Machines

In [27]:
evaluation_df["relevance"] = ["Relevant"] * len(evaluation_df)

evaluation_df["grounded"] = ["Grounded"] * len(evaluation_df)

evaluation_df["failure_case"] = ["None"] * len(evaluation_df)

evaluation_df["mitigation"] = ["None"] * len(evaluation_df)

evaluation_df

,question_id,question,answer,sources,relevance,grounded,failure_case,mitigation
0,1,What is machine learning?,Machine Learning (ML) is a branch of Artifici...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
1,2,What are the main types of machine learning?,The main types of Machine Learning (ML) are S...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
2,3,What is supervised learning?,Supervised Learning is a type of Machine Lear...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
3,4,What is linear regression?,Linear Regression is a statistical method use...,Session 6 - Linear Regresssion.pdf,Relevant,Grounded,None,None
4,5,What is the hypothesis function in linear regr...,The hypothesis function in linear regression ...,"Session 6 - Linear Regresssion.pdf, Session 7 ...",Relevant,Grounded,None,None
5,6,What is the cost function?,The cost function is a measure that quantifie...,"Session 6 - Linear Regresssion.pdf, Session 7 ...",Relevant,Grounded,None,None
6,7,What is multiple linear regression?,Multiple linear regression is a type of linea...,Session 6 - Linear Regresssion.pdf,Relevant,Grounded,None,None
7,8,What is logistic regression?,Logistic Regression is a statistical modeling...,Session 7 Ml -logistic regression.pdf,Relevant,Grounded,None,None
8,9,What is the sigmoid function?,The sigmoid function is a mathematical functi...,Session 7 Ml -logistic regression.pdf,Relevant,Grounded,None,None
9,10,What is classification?,Classification is a type of machine learning ...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None


In [30]:
out_of_domain_question = "Who won the FIFA World Cup in 2022?"

answer, sources = generate_answer(
    out_of_domain_question
)

print("Question:", out_of_domain_question)
print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source["source"])

Question: Who won the FIFA World Cup in 2022?

Answer:
 I couldn't find this information in the provided course materials. The course material discusses Artificial Intelligence, Generative Models, Supervised Learning, Regression, and Classification, but there is no mention of the FIFA World Cup or its results from 2022. To find out who won the FIFA World Cup in 2022, you may want to check a reliable sports news source or the official FIFA website.

Sources:
- Session 3 intro to ML.pdf
- Session 3 intro to ML.pdf
- Session 3 intro to ML.pdf


In [31]:
evaluation_results.append({
    "question_id": 11,
    "question": out_of_domain_question,
    "answer": answer,
    "sources": ", ".join(
        sorted(set(source["source"] for source in sources))
    )
})

In [32]:
evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question_id,question,answer,sources
0,1,What is machine learning?,Machine Learning (ML) is a branch of Artifici...,Session 3 intro to ML.pdf
1,2,What are the main types of machine learning?,The main types of Machine Learning (ML) are S...,Session 3 intro to ML.pdf
2,3,What is supervised learning?,Supervised Learning is a type of Machine Lear...,Session 3 intro to ML.pdf
3,4,What is linear regression?,Linear Regression is a statistical method use...,Session 6 - Linear Regresssion.pdf
4,5,What is the hypothesis function in linear regr...,The hypothesis function in linear regression ...,"Session 6 - Linear Regresssion.pdf, Session 7 ..."
5,6,What is the cost function?,The cost function is a measure that quantifie...,"Session 6 - Linear Regresssion.pdf, Session 7 ..."
6,7,What is multiple linear regression?,Multiple linear regression is a type of linea...,Session 6 - Linear Regresssion.pdf
7,8,What is logistic regression?,Logistic Regression is a statistical modeling...,Session 7 Ml -logistic regression.pdf
8,9,What is the sigmoid function?,The sigmoid function is a mathematical functi...,Session 7 Ml -logistic regression.pdf
9,10,What is classification?,Classification is a type of machine learning ...,Session 3 intro to ML.pdf


In [34]:
evaluation_df["relevance"] = ["Relevant"] * 10 + ["Not Relevant"]

evaluation_df["grounded"] = ["Grounded"] * 10 + ["Not Grounded"]

evaluation_df["failure_case"] = (
    ["None"] * 10
    + ["Out-of-domain question"]
)

evaluation_df["mitigation"] = (
    ["None"] * 10
    + ["Return a fallback response when information is unavailable"]
)
evaluation_df

,question_id,question,answer,sources,relevance,grounded,failure_case,mitigation
0,1,What is machine learning?,Machine Learning (ML) is a branch of Artifici...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
1,2,What are the main types of machine learning?,The main types of Machine Learning (ML) are S...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
2,3,What is supervised learning?,Supervised Learning is a type of Machine Lear...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None
3,4,What is linear regression?,Linear Regression is a statistical method use...,Session 6 - Linear Regresssion.pdf,Relevant,Grounded,None,None
4,5,What is the hypothesis function in linear regr...,The hypothesis function in linear regression ...,"Session 6 - Linear Regresssion.pdf, Session 7 ...",Relevant,Grounded,None,None
5,6,What is the cost function?,The cost function is a measure that quantifie...,"Session 6 - Linear Regresssion.pdf, Session 7 ...",Relevant,Grounded,None,None
6,7,What is multiple linear regression?,Multiple linear regression is a type of linea...,Session 6 - Linear Regresssion.pdf,Relevant,Grounded,None,None
7,8,What is logistic regression?,Logistic Regression is a statistical modeling...,Session 7 Ml -logistic regression.pdf,Relevant,Grounded,None,None
8,9,What is the sigmoid function?,The sigmoid function is a mathematical functi...,Session 7 Ml -logistic regression.pdf,Relevant,Grounded,None,None
9,10,What is classification?,Classification is a type of machine learning ...,Session 3 intro to ML.pdf,Relevant,Grounded,None,None


In [35]:
evaluation_path = PROJECT_DIR / "evaluation_results.csv"

evaluation_df.to_csv(evaluation_path, index=False)

print(f"Evaluation saved to: {evaluation_path}")

Evaluation saved to: d:\🎓 Project StudyMate — RAG-Powered University Assistant\evaluation_results.csv


In [36]:
import json

config = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_device": device,
    "vector_database": "ChromaDB",
    "collection_name": "studymate_documents",
    "chunk_size": 800,
    "chunk_overlap": 150,
    "retrieval_k": 3,
    "llm": "mistral"
}

config_path = (
    PROJECT_DIR
    / "backend"
    / "data"
    / "vector_store"
    / "config.json"
)

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")

Config saved to: d:\🎓 Project StudyMate — RAG-Powered University Assistant\backend\data\vector_store\config.json


In [37]:
print("Vector store:", VECTOR_DB_DIR)
print("Stored chunks:", collection.count())

Vector store: d:\🎓 Project StudyMate — RAG-Powered University Assistant\backend\data\vector_store
Stored chunks: 45
